<a href="https://colab.research.google.com/github/Semanur97/Sam3_Usage/blob/main/notebooks/colab_coco_t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM3 fine-tuning — Coco_T bubble segmentation

Kod `coco_t/` klasöründe (GitHub). Bu notebook sadece kodu çeker ve GPU'da çalıştırır.
Kodda değişiklik GitHub'da yapılır; burada sadece **2. hücreyi** tekrar çalıştırman yeterli.

Runtime → Change runtime type → **GPU (L4)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Kodu GitHub'dan çek (ilk seferde clone, sonra pull) + kurulum
import os
REPO = '/content/Sam3_Usage'
if not os.path.exists(REPO):
    !git clone https://github.com/Semanur97/Sam3_Usage.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!git log --oneline -1
!pip install -q -r requirements.txt

# pull sonrası kod değişiklikleri runtime'ı yeniden başlatmadan yüklensin
%load_ext autoreload
%autoreload 2

In [ ]:
# Hugging Face girişi (Colab Secrets'ta 'HFtoken' kayıtlı olmalı)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HFtoken'))

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UYARI: GPU yok!')

## Ayarlar
Varsayılanlar `coco_t/config.py` içinde. Burada istediğini değiştir.

In [ ]:
from coco_t.config import Config

cfg = Config(
    epochs=10,
    lr=1e-4,
    max_side=512,
    box_chunk=16,   # OOM olursa 8 veya 4 yap
)
cfg

## Eğitim
Epoch 0 = fine-tune öncesi (zero-shot) doğrulama. En iyi decoder → `save_dir/coco_t_sam3_decoder_best.pt`

In [ ]:
from coco_t.train import train
history = train(cfg)

## Zero-shot vs fine-tuned (aynı GT kutularıyla, tüm val seti)
Sonuçlar Drive'a kaydedilir: `coco_t_eval.json`, `coco_t_eval_per_image.csv`, `coco_t_sam3_comparison.png`

In [ ]:
from coco_t.evaluate import compare
results = compare(cfg)